# Exp. 1 Ablation: Hidden Spatial Confounder

Clam assumes no hidden confounding. This ablation adds a dial $\gamma$ that makes this
assumption progressively more wrong, and measures how quickly the estimates degrade.

Changes relative to Exp. 1 (data generation only, Clam itself is unchanged):

1. A hidden, spatially smooth field $u$ that Clam never sees.
2. It affects outcomes: $y_{ij} = f(t_{ij}, c_{ij}) + \gamma \, u_{ij} + \epsilon_{ij}$.
3. It affects treatment: the half of the regions with the *highest* average $u$ is treated
   (instead of a random half).

At $\gamma = 0$ this is exactly Exp. 1. As $\gamma$ grows, treated regions have higher outcomes
*because of* $u$, and since Clam cannot see $u$, it credits the treatment.
The predicted bias of the average effect is
$\gamma \cdot (\bar{u}_{\text{treated}} - \bar{u}_{\text{control}})$, shown as a dashed line below.

## Setup (identical to Exp. 1)

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

# Parameters, identical to Exp. 1 (political campaigning)
SEED = 42
ROW = 100          # number of regions, arranged on a 10 x 10 grid
COL = 16           # subregions per region, arranged on a 4 x 4 grid
N_SWAPS = 100      # random pair swaps in the context matrix
NOISE_STD = 0.1    # observation noise on the subregional outcomes
N_EPOCHS = 4000
LR = 1e-3
WEIGHT_DECAY = 1e-3
HIDDEN_DIM = 32
DROPOUT = 0.05


def set_all_seeds(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)


## Data generation with the confounder dial $\gamma$

In [ ]:
def true_f(t, c):
    """Ground-truth mechanism: outcome is 1 if untreated,
    and a piecewise-linear function of the context if treated."""
    anchor_x = np.array([-100, -2, 0, 1, 100], dtype=float)
    anchor_y = np.array([2, 2, 1, 0, 0], dtype=float)
    return np.where(t == 0, 1.0, np.interp(c, anchor_x, anchor_y))


def create_context_matrix():
    """Row-centered Gaussian context with random swaps,
    so the regional mean context is uninformative by construction."""
    C = np.random.normal(size=(ROW, COL))
    C = C - C.mean(axis=1, keepdims=True)
    for _ in range(N_SWAPS):
        r1, c1 = np.random.randint(ROW), np.random.randint(COL)
        r2, c2 = np.random.randint(ROW), np.random.randint(COL)
        C[r1, c1], C[r2, c2] = C[r2, c2], C[r1, c1]
    return C


def to_grid(M):
    """Rearrange a (ROW, COL) region-by-subregion matrix into its (40, 40) spatial map."""
    R, S = int(np.sqrt(ROW)), int(np.sqrt(COL))
    grid = np.zeros((R * S, R * S))
    for i in range(ROW):
        for j in range(COL):
            grid[(i // R) * S + (j // S), (i % R) * S + (j % S)] = M[i, j]
    return grid


def from_grid(grid):
    """Inverse of to_grid: (40, 40) spatial map back to a (ROW, COL) matrix."""
    R, S = int(np.sqrt(ROW)), int(np.sqrt(COL))
    M = np.zeros((ROW, COL))
    for i in range(ROW):
        for j in range(COL):
            M[i, j] = grid[(i // R) * S + (j // S), (i % R) * S + (j % S)]
    return M


from scipy.ndimage import gaussian_filter


def make_confounder():
    """Hidden, spatially smooth field u with zero mean and unit variance."""
    u = gaussian_filter(np.random.normal(size=(40, 40)), sigma=3)
    return from_grid((u - u.mean()) / u.std())


def generate_data(gamma, seed):
    """Exp. 1 data, plus a hidden confounder of strength gamma."""
    set_all_seeds(seed)
    C = create_context_matrix()
    U = make_confounder()

    # Confounded treatment: the half of the regions with the highest average u is treated.
    T = np.zeros((ROW, COL))
    T[np.argsort(U.mean(axis=1))[ROW // 2:]] = 1.0

    Y = true_f(T, C) + gamma * U + np.random.normal(scale=NOISE_STD, size=(ROW, COL))
    E_true = true_f(T, C) - true_f(np.zeros_like(T), C)
    return T, C, U, Y.mean(axis=1), E_true


## Model and Clam training (identical to Exp. 1, no per-epoch bookkeeping)

In [ ]:
class SmallMLP(nn.Module):
    """f_theta(t, c): two inputs, one scalar output. Same architecture as in Exp. 1."""

    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, HIDDEN_DIM), nn.ReLU(), nn.Dropout(DROPOUT),
            nn.Linear(HIDDEN_DIM, HIDDEN_DIM), nn.ReLU(), nn.Dropout(DROPOUT),
            nn.Linear(HIDDEN_DIM, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)


def features(T_mat, C_mat):
    return torch.tensor(np.stack([T_mat, C_mat], axis=-1), dtype=torch.float32).reshape(-1, 2)


def train_clam(T, C, Y_agg, seed):
    """Train Clam on the aggregate loss and return the LOCATE estimate."""
    torch.manual_seed(seed)
    model = SmallMLP()
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    X = features(T, C)
    Y_agg_t = torch.tensor(Y_agg, dtype=torch.float32)

    model.train()
    for epoch in range(N_EPOCHS):
        pred = model(X).reshape(ROW, COL)
        loss = torch.mean((pred.mean(dim=1) - Y_agg_t) ** 2)
        opt.zero_grad()
        loss.backward()
        opt.step()

    model.eval()
    with torch.no_grad():
        pred = model(X).reshape(ROW, COL)
        pred0 = model(features(np.zeros_like(T), C)).reshape(ROW, COL)
    return (pred - pred0).numpy()


## Sweep the confounder strength

In [ ]:
GAMMAS = [0.0, 0.25, 0.5, 1.0, 2.0]
SEEDS = [0, 1, 2, 3, 4]

results = []
for gamma in GAMMAS:
    for seed in SEEDS:
        T, C, U, Y_agg, E_true = generate_data(gamma, seed)
        E_pred = train_clam(T, C, Y_agg, seed)

        treated = T == 1
        results.append({
            "gamma": gamma,
            "seed": seed,
            "locate_mae": np.abs(E_pred - E_true).mean(),
            "att_true": E_true[treated].mean(),          # true avg. effect among the treated
            "att_est": E_pred[treated].mean(),           # Clam's estimate of it
            "delta_u": U[treated].mean() - U[~treated].mean(),
        })
        print(f"gamma={gamma:<5} seed={seed} | LOCATE MAE={results[-1]['locate_mae']:.4f} "
              f"| ATT est={results[-1]['att_est']:.3f} (true {results[-1]['att_true']:.3f})")

import pandas as pd
df = pd.DataFrame(results)
df.groupby("gamma")[["locate_mae", "att_est", "att_true"]].mean()


## Results

In [ ]:
summary = df.groupby("gamma").agg(["mean", "std"])
gammas = summary.index.values

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

# (a) Estimation error vs. confounder strength
ax1.errorbar(gammas, summary["locate_mae"]["mean"], yerr=summary["locate_mae"]["std"],
             marker="o", capsize=3)
ax1.set_xlabel(r"Confounder strength $\gamma$")
ax1.set_ylabel("LOCATE MAE")
ax1.set_title("Error grows with the assumption violation")

# (b) Estimated vs. predicted average effect among the treated
att_true = df["att_true"].mean()
delta_u = df["delta_u"].mean()
ax2.errorbar(gammas, summary["att_est"]["mean"], yerr=summary["att_est"]["std"],
             marker="o", capsize=3, label="Clam estimate")
ax2.plot(gammas, att_true + np.array(gammas) * delta_u, "k--",
         label=r"Predicted: true + $\gamma\,\Delta\bar{u}$")
ax2.axhline(att_true, color="gray", lw=1, label="True effect")
ax2.set_xlabel(r"Confounder strength $\gamma$")
ax2.set_ylabel("Avg. treatment effect (treated)")
ax2.set_title("Bias matches the closed-form prediction")
ax2.legend()

plt.tight_layout()
plt.show()
